# 3D reconstruction via colmap and nerfstudio splatfacto

* this currently only works in linux. expects a conda environment setup. tested with ubuntu 24.04.3 LTS

## setup

In [1]:
import os
import sys
from pathlib import Path

In [2]:
# Locate dt4ag_config.py (it lives in the pipeline/ directory, one level above
# notebooks/) and import the config loader. Walking upward keeps this working
# whether the notebook runs from the repo checkout or from a copy sitting next
# to the data.
def _find_pipeline_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dt4ag_config.py").is_file():
            return candidate
    raise FileNotFoundError(
        f"could not find dt4ag_config.py by walking up from {here}. Run this "
        "notebook from inside the jsps-dt4ag repo, or add the directory "
        "containing dt4ag_config.py to PYTHONPATH."
    )


pipeline_root = _find_pipeline_root()
if str(pipeline_root) not in sys.path:
    sys.path.insert(0, str(pipeline_root))

from dt4ag_config import ConfigError, find_config, load_config

print("pipeline_root:", pipeline_root)

In [3]:
print(os.getcwd())

/media/alex/T5Red/DT-data/notebooks


In [4]:
#ensure colmap is installed and check version
!colmap

COLMAP 3.12.0 -- Structure-from-Motion and Multi-View Stereo
(Commit e7e89eb0 on 2025-06-30 with CUDA)

Usage:
  colmap [command] [options]
Documentation:
  https://colmap.github.io/
Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...
Available commands:
  help
  gui
  automatic_reconstructor
  bundle_adjuster
  color_extractor
  database_cleaner
  database_creator
  database_merger
  delaunay_mesher
  exhaustive_matcher
  feature_extractor
  feature_importer
  hierarchical_mapper
  image_deleter
  image_filterer
  image_rectifier
  image_registrator
  image_undistorter
  image_undistorter_standalone
  mapper
  matches_im

In [5]:
# Load the run configuration.
#
# Everything that used to be typed into these cells by hand (data root, dataset
# selection, colmap flags, training and export settings) now comes from an INI
# file. See configs/README.md for what every key does.
#
# Precedence: the DT4AG_CONFIG environment variable, else configs/example.ini
# found by walking up from pipeline_root. To pick a config from here:
#     os.environ["DT4AG_CONFIG"] = "/path/to/my-run.ini"
config_path = find_config(start=pipeline_root)
cfg = load_config(config_path)
print(cfg.describe())

project_id: run_260224-01-312


In [6]:
# Run identity.
#
# run_date, todays_run_count and colmap_ver used to be typed by hand every run.
# All three are now derived: the date from today's clock, the count by
# inspecting the run directories already present for this dataset, the COLMAP
# version by invoking `colmap`. Any of them can still be pinned in [run] to
# reproduce an old run id.
project_id = cfg.make_run_id()
print('project_id:', project_id)

# One row per run id issued. Re-running this cell appends another row.
print('run log:', cfg.append_run_log(project_id))

# Root paths, all hanging off [paths] data_root.
base_dir = cfg.data_root
colmap_dir = cfg.colmap_dir       # the colmap workspace
dataset_dir = cfg.datasets_dir    # the dataset (2d raw images) dir
export_dir = cfg.exports_dir      # nerfstudio exports (like .ply splats)
output_dir = cfg.outputs_dir      # nerfstudio outputs (like config.yml)
export_3dgs = cfg.export_3dgs

In [7]:
# Use dictionary to store path objects
paths_dict = {
    "data_root": base_dir,
    "colmap_dir": colmap_dir,
    "dataset_dir": dataset_dir,
    "export_dir": export_dir,
    "output_dir": output_dir,
}

In [8]:
for key, value in paths_dict.items():
    print(f"Key: {key}, Value: {value}", "Type:", type(value))


Key: base_dir_abs, Value: /media/alex/T5Red/DT-data Type: <class 'pathlib.PosixPath'>
Key: base_dir_rel, Value: /media/alex/T5Red/DT-data Type: <class 'pathlib.PosixPath'>
Key: colmap_dir, Value: /media/alex/T5Red/DT-data/colmap Type: <class 'pathlib.PosixPath'>
Key: dataset_dir, Value: /media/alex/T5Red/DT-data/datasets Type: <class 'pathlib.PosixPath'>
Key: export_dir, Value: /media/alex/T5Red/DT-data/exports Type: <class 'pathlib.PosixPath'>
Key: output_dir, Value: /media/alex/T5Red/DT-data/outputs Type: <class 'pathlib.PosixPath'>


## colmap related commands

In [10]:
# list dataset contents via bash
!ls -ll $dataset_dir

total 1536
drwxr-xr-x 13 alex alex 262144 Jan 17 13:58 alex-home
drwxr-xr-x  7 alex alex 262144 Aug 13  2025 archive_1
drwxr-xr-x  5 alex alex 262144 Aug 13  2025 examples
drwxr-xr-x  3 alex alex 262144 Jul 30  2025 kyushu_soybean
drwxr-xr-x  5 alex alex 262144 Jan  5 21:42 sam3-testing
drwxr-xr-x  8 alex alex 262144 Jan 16 21:26 tanashi


In [11]:
# list dataset contents via python
contents = list(Path(dataset_dir).iterdir())
for item in contents: print(item)

/media/alex/T5Red/DT-data/datasets/examples
/media/alex/T5Red/DT-data/datasets/kyushu_soybean
/media/alex/T5Red/DT-data/datasets/tanashi
/media/alex/T5Red/DT-data/datasets/alex-home
/media/alex/T5Red/DT-data/datasets/sam3-testing
/media/alex/T5Red/DT-data/datasets/archive_1


In [12]:
# Resolve the dataset image directory.
#
# This used to be four chained loops doing `if str(subdir_id) in str(path)`.
# They matched on substrings and never broke out of the loop, so when two
# sibling directories both matched, the LAST one silently won. The path is now
# stated explicitly as [dataset] images_subpath and its existence is checked
# when the config loads.
colmap_reconstruction_images_path = cfg.images_path

print('images_subpath (relative to datasets dir):', cfg.images_rel)
print('colmap_reconstruction_images_path:', colmap_reconstruction_images_path)
print('path exists:', colmap_reconstruction_images_path.exists())

In [13]:
# Sanity check. An empty image directory is a silent-failure trap: COLMAP will
# happily run on nothing and produce nothing, and the upstream masking script
# has a known no-op mode that leaves exactly that behind.
image_suffixes = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
image_files = sorted(
    p for p in colmap_reconstruction_images_path.iterdir()
    if p.is_file() and p.suffix.lower() in image_suffixes
)
print('image files found:', len(image_files))
if not image_files:
    raise RuntimeError(
        f'no image files in {colmap_reconstruction_images_path}. Check '
        f'[dataset] images_subpath in {cfg.source}'
    )
print('first:', image_files[0].name, '| last:', image_files[-1].name)

found it
subdir_a /media/alex/T5Red/DT-data/datasets/tanashi <class 'pathlib.PosixPath'>


In [14]:
# list the chosen image directory via bash
!ls -ll $colmap_reconstruction_images_path

total 1536
drwxr-xr-x 8 alex alex 262144 Aug 13  2025 202410_tanashi_arabidopsis
drwxr-xr-x 3 alex alex 262144 Aug 13  2025 202410_tanashi_tomato
drwxr-xr-x 2 alex alex 262144 Jan 16 21:25 logs
drwxr-xr-x 3 alex alex 262144 Jan 16 21:26 masks
drwxr-xr-x 9 alex alex 262144 Dec 18 13:56 test_251-tomato
drwxr-xr-x 8 alex alex 262144 Aug 13  2025 wheat_head_s4w3


In [24]:
# Path components used to name the colmap workspace, the nerfstudio output
# directory and the export file. `dataset_rel` mirrors the dataset's path
# across the colmap/ and outputs/ trees.
#
# This used to walk four `.parent` levels and so assumed a depth-4 hierarchy,
# which is why the dataset-selection cell carried an "uncomment this line if
# there is no fourth level" escape hatch. Taking the configured relative path
# directly is depth-agnostic and needs no escape hatch.
dataset_rel = cfg.images_rel
images_dir_name = colmap_reconstruction_images_path.name
images_parent_name = colmap_reconstruction_images_path.parent.name

print('dataset_rel:', dataset_rel)
print('images_dir_name:', images_dir_name)
print('images_parent_name:', images_parent_name)

test_251128 masked-images test_251-tomato tanashi tanashi/test_251-tomato/masked-images/test_251128


In [26]:
colmap_reconstruction_workspace_path = cfg.colmap_workspace(project_id)
print('colmap_reconstruction_workspace_path', colmap_reconstruction_workspace_path)
print('path exists:', colmap_reconstruction_workspace_path.exists())
colmap_reconstruction_workspace_path.mkdir(parents=True, exist_ok=True)
print('path exists:', colmap_reconstruction_workspace_path.exists())

colmap_reconstruction_workspace_path /media/alex/T5Red/DT-data/colmap/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312
path exists: False
path exists: True


### execute colmap automatic reconstruction

In [27]:
# `auto` infers from the image directory name: a name mentioning "frames" came
# from a video, anything else is treated as individual images. Override with
# [colmap] data_type.
colmap_data_type = cfg.resolve_colmap_data_type()

print('colmap_data_type', colmap_data_type)
print('colmap_reconstruction_images_path', colmap_reconstruction_images_path)
print('colmap_reconstruction_workspace_path', colmap_reconstruction_workspace_path)

colmap_data_type individual
colmap_reconstruction_images_path /media/alex/T5Red/DT-data/datasets/tanashi/test_251-tomato/masked-images/test_251128
colmap_reconstruction_workspace_path /media/alex/T5Red/DT-data/colmap/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312


In [28]:
colmap_cmd = ' '.join(filter(None, [
    'colmap automatic_reconstructor',
    f'--workspace_path {colmap_reconstruction_workspace_path}',
    f'--image_path {colmap_reconstruction_images_path}',
    f'--data_type {colmap_data_type}',
    f'--single_camera {cfg.colmap_single_camera}',
    f'--single_camera_per_folder {cfg.colmap_single_camera_per_folder}',
    f'--dense {cfg.colmap_dense}',
    cfg.colmap_extra_args,
]))
print(colmap_cmd)
!{colmap_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'colmap exited {globals()["_exit_code"]}: {colmap_cmd}')

I20260224 22:53:48.868380 138263152332800 misc.cc:44] 
Feature extraction
I20260224 22:53:48.869211 138262832205824 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869292 138262823813120 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869384 138262815420416 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869462 138262807027712 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869545 138262798635008 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869625 138262513446912 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869721 138262505054208 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:48.869802 138262496661504 sift.cc:717] Creating Covariant SIFT CPU feature extractor
I20260224 22:53:53.089287 138262488268800 feature_extraction.cc:259] Processed file [1/120]
I20260224 22:53:53.089335 

## run nerfstudio commands

### prep for ns-process-data

In [29]:
scene = cfg.scene_type                       # 'images' or 'video'
ns_images_dir = colmap_reconstruction_images_path
ns_video_path = cfg.video_path

print('scene:', scene)
print('ns_images_dir:', ns_images_dir, ns_images_dir.exists())
print('ns_video_path:', ns_video_path or '(unset)')

scene: images
ns_images_dir: /media/alex/T5Red/DT-data/datasets/tanashi/test_251-tomato/masked-images/test_251128 True
ns_video_path: 


In [30]:
ns_colmap_dir = colmap_reconstruction_workspace_path
print('ns_colmap_dir:',ns_colmap_dir,ns_colmap_dir.exists())

ns_colmap_dir: /media/alex/T5Red/DT-data/colmap/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312 True


In [31]:
!ls $ns_colmap_dir

database.db  sparse


### run ns-process-data

In [32]:
if scene == 'images':
    ns_process_cmd = ' '.join(filter(None, [
        'ns-process-data images',
        f'--data {ns_images_dir}',
        f'--output-dir {ns_colmap_dir}',
        '--skip-colmap' if cfg.skip_colmap else '',
        f'--colmap-model-path {cfg.colmap_model_path}',
    ]))
elif scene == 'video':
    # Not currently exercised by this pipeline.
    ns_process_cmd = (
        f'ns-process-data video --data {ns_video_path} --output-dir {ns_colmap_dir}'
    )
else:
    raise ValueError(f'unsupported [nerfstudio] scene_type: {scene!r}')

print(ns_process_cmd)
!{ns_process_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-process-data exited {globals()["_exit_code"]}')
print("Data Processing Succeeded!")

[23:05:02] 🎉 Done copying images with prefix 'frame_'.                                        ]8;id=244804;file:///home/alex/nerfstudio-1.1.5/nerfstudio/process_data/process_data_utils.py\process_data_utils.py]8;;\:]8;id=960437;file:///home/alex/nerfstudio-1.1.5/nerfstudio/process_data/process_data_utils.py#348\348]8;;\
(    ● ) Copying images...
/media/alex/T5Red/DT-data/colmap/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312/sparse/0
{1: Camera(id=1, model='SIMPLE_RADIAL', width=5184, height=3456, params=array([1.23659181e+04, 2.59200000e+03, 
1.72800000e+03, 7.54502124e-03])), 2: Camera(id=2, model='SIMPLE_RADIAL', width=5184, height=3456, 
params=array([1.22883190e+04, 2.59200000e+03, 1.72800000e+03, 2.98385316e-02])), 3: Camera(id=3, model='SIMPLE_RADIAL', 
width=5184, height=3456, params=array([1.22495099e+04, 2.59200000e+03, 1.72800000e+03, 2.17318873e-02])), 4: 
Camera(id=4, model='SIMPLE_RADIAL', width=5184, height=3456, params=array([ 7.17819738e+03

### prep for ns-train

In [33]:
!ls $ns_colmap_dir

database.db  images_2  images_8  sparse_pc.ply
images	     images_4  sparse	 transforms.json


In [34]:
# prep output dir
ns_output_dir = cfg.output_parent
print('ns_output_dir:', ns_output_dir, ns_output_dir.exists())

/media/alex/T5Red/DT-data/outputs
/media/alex/T5Red/DT-data/outputs/tanashi/test_251-tomato/masked-images/test_251128 True


### run ns-train

In [36]:
num_steps = cfg.max_num_iterations   # [train] max_num_iterations, splatfacto default is 30000
ns_train_cmd = ' '.join([
    f'ns-train {cfg.train_method}',
    f'--data {ns_colmap_dir}',
    f'--pipeline.model.use_scale_regularization {cfg.use_scale_regularization}',
    f'--pipeline.model.background_color {cfg.background_color}',
    f'--output-dir {ns_output_dir}',
    f'--viewer.quit-on-train-completion {cfg.quit_on_train_completion}',
    f'--max-num-iterations {num_steps}',
    f'--logging.local-writer.max-log-size {cfg.max_log_size}',
])
print(ns_train_cmd)
!{ns_train_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-train exited {globals()["_exit_code"]}')

/home/alex/nerfstudio-1.1.5/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/home/alex/nerfstudio-1.1.5/nerfstudio/field_components/activations.py:39: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, g):
[23:11:52] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=152515;file:///home/alex/nerfstudio-1.1.5/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=698219;file:///home/alex/nerfstudio-1.1.5/nerfstudio/scripts/train.py#230\230]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=Posi

## export splat

### somehow get the config.yml path!!

In [38]:
# nerfstudio names the experiment after the --data directory, which is the
# colmap workspace, which is named after the run id.
config_yml_parent_path = ns_output_dir / project_id / cfg.train_method
print('config_yml_parent_path', config_yml_parent_path, config_yml_parent_path.exists())

config_yml_parent_path /media/alex/T5Red/DT-data/outputs/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312/splatfacto True


In [39]:
# Timestamped checkpoint directories sort chronologically; take the newest.
checkpoint_dirs = sorted(p for p in config_yml_parent_path.iterdir() if p.is_dir())
if not checkpoint_dirs:
    raise RuntimeError(f'no training run directories under {config_yml_parent_path}')

for path in checkpoint_dirs:
    print(path.name)

config_yml_path = checkpoint_dirs[-1] / 'config.yml'
if not config_yml_path.is_file():
    raise RuntimeError(f'no config.yml at {config_yml_path}')

print('config_yml_path', config_yml_path)

2026-02-24_230519
2026-02-24_231152
config_yml_path /media/alex/T5Red/DT-data/outputs/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312/splatfacto/2026-02-24_231152/config.yml
True True


In [40]:
print('dataset_rel:', dataset_rel)
print('images_dir_name:', images_dir_name, '| images_parent_name:', images_parent_name)
print('project_id:', project_id)

tanashi test_251-tomato masked-images test_251128
tanashi/test_251-tomato/masked-images/test_251128 run_260224-01-312


In [41]:
# Export filename records what produced it: dataset, run id, platform, env,
# iteration count and colmap data type. The labels come from [export].
ns_export_dir = config_yml_path.parent / 'exports'
ns_export_filename = '_'.join([
    str(images_parent_name),
    str(project_id),
    'splat',
    cfg.platform_label,
    cfg.env_label,
    f'{num_steps}steps',
    colmap_data_type,
]) + '.ply'
print('ns_export_dir:', ns_export_dir)
print('ns_export_filename:', ns_export_filename)

In [42]:
ns_export_fullpath = ns_export_dir / ns_export_filename

print ("Export splat to:",ns_export_fullpath)
print('checkpoint path exists:',ns_export_fullpath.parent.parent.exists())

Export splat to: /media/alex/T5Red/DT-data/outputs/tanashi/test_251-tomato/masked-images/test_251128/run_260224-01-312/splatfacto/2026-02-24_231152/exports/masked-images_run_260224-01-312_splat_ubuntu_ns-l-oci_30000steps_individual.ply
checkpoint path exists: True


### run ns-export

In [43]:
ns_export_cmd = ' '.join([
    f'ns-export {cfg.export_format}',
    f'--load-config {config_yml_path}',
    f'--output-dir {ns_export_dir}',
    f'--output-filename {ns_export_filename}',
])
print(ns_export_cmd)
!{ns_export_cmd}
if globals().get('_exit_code', 0):
    raise RuntimeError(f'ns-export exited {globals()["_exit_code"]}')
print("Export successful!")

/home/alex/nerfstudio-1.1.5/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/home/alex/nerfstudio-1.1.5/nerfstudio/field_components/activations.py:39: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, g):
/home/alex/miniconda3/envs/ns-l-oci/lib/python3.10/site-packages/torchmetrics/functional/image/lpips.py:332: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_

### export 3dgs to point cloud (note this increases file size like 1000X!!)

In [48]:
# Optional: convert the exported 3DGS splat to a point cloud.
# input path example:     <data_root>/outputs/<dataset_rel>/<run_id>/<method>/<timestamp>/exports/<name>.ply
# transform path example: <data_root>/colmap/<dataset_rel>/<run_id>/transforms.json
a_3dgs_input_path = ns_export_fullpath
a_3dgs_transforms_path = ns_colmap_dir
a_3dgs_output_path = str(a_3dgs_input_path.parent / a_3dgs_input_path.stem) + '_3dgs-to-pc.ply'

print(a_3dgs_input_path)
print(a_3dgs_transforms_path)
print(a_3dgs_output_path)

/media/alex/T5Red/DT-data/outputs/alex-home/gopro/test1/sharp-frames/run_260114-03-312/splatfacto/2026-01-14_170326/exports/test1_run_260114-03-312_splat_ubuntu_ns-l-oci_30000steps_video.ply /media/alex/T5Red/DT-data/colmap/alex-home/gopro/test1/sharp-frames/run_260114-03-312 /media/alex/T5Red/DT-data/outputs/alex-home/gopro/test1/sharp-frames/run_260114-03-312/splatfacto/2026-01-14_170326/exports/test1_run_260114-03-312_splat_ubuntu_ns-l-oci_30000steps_video_3dgs-to-pc.ply


In [49]:
if export_3dgs:
    gauss_to_pc = os.path.expanduser(cfg.gauss_to_pc_script)
    gauss_to_pc_cmd = ' '.join([
        f'python {gauss_to_pc}',
        f'--input_path {a_3dgs_input_path}',
        f'--transform_path {a_3dgs_transforms_path}',
        f'--output_path {a_3dgs_output_path}',
    ])
    print(gauss_to_pc_cmd)
    !{gauss_to_pc_cmd}
    if globals().get('_exit_code', 0):
        raise RuntimeError(f'gauss_to_pc exited {globals()["_exit_code"]}')